# Menezes-Almeida et al. (202x) — analysis code


**Disclaimer:** this repository contains only the analysis code. It does
**not** include the underlying data. To reproduce an
analysis, obtain the relevant file(s) from the Supplementary Data and set
the corresponding input path/filename at the top of that section
accordingly.


**Sections**

1. Setup
2. Phanerozoic regional context timeline — Figure 2 (main text)
3. Mean track length (MTL) histograms per sample — Supplementary Fig. S2
4. AFT age vs. elevation — Supplementary Fig. S3
5. Boomerang plot (MTL vs. central age), measured and projected — Supplementary Fig. S4
6. Expected t–T paths by crustal domain and MTL cluster — Figure 5 (main text)
7. QTQt cooling-rate and exhumation-rate calculation, per sample and combined — Supplementary Table S3 / Fig. S6


## 1. Setup

In [ ]:
import os
import re
import glob
import csv
import textwrap

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

fm.findSystemFonts(fontpaths=None, fontext="ttf")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Montserrat", "DejaVu Sans", "Arial"]


## 2. Phanerozoic regional context timeline — Figure 2 (main text)

In [ ]:
import os
import re
import textwrap

import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

SAMPLES_FILE = "literature_aft_data.xlsx"
SHEET_NAME = "Literature"

COLUMN_ALIASES = {
    "Age (Ma)": "age_ma",
    "Central age (Ma)": "age_ma",
    "AFT_age": "age_ma",
    "Subdomain": "domain",
    "Craton": "domain",
    "Province": "domain",
    "reference": "source",
    "Reference": "source",
    "Sample": "sample_id",
}

DOMAIN_ALIASES = {
    "sao francisco craton": "São Francisco Craton",
    "sao francisco": "São Francisco Craton",
    "são francisco": "São Francisco Craton",
    "sfc": "São Francisco Craton",
    "craton": "São Francisco Craton",
    "borborema province": "Borborema Province",
    "borborema": "Borborema Province",
    "sergipano belt": "Borborema Province",
    "sergipano": "Borborema Province",
    "peal": "Borborema Province",
}
VIOLIN_ORDER = ["São Francisco Craton", "Borborema Province"]
VIOLIN_COLORS = {"São Francisco Craton": "#c0392b", "Borborema Province": "#2f7d5c"}

PHANEROZOIC_EVENTS = [
    dict(label="Pampean & Tilcaric orogenies", age_start=540, age_end=520,
         category="tectonic", ref="Cawood, 2005; Cawood and Buchan, 2007"),
    dict(label="Famatinian orogeny", age_start=490, age_end=440,
         category="tectonic", ref="Cawood, 2005; Anderson et al., 2021"),
    dict(label="Ocloyic orogeny", age_start=452, age_end=420,
         category="tectonic", ref="Cawood, 2005; Cawood and Buchan, 2007"),
    dict(label="Gondwanide orogeny", age_start=320, age_end=250,
         category="tectonic", ref="Cawood, 2005; Anderson et al., 2021"),
    dict(label="Palmares Fm. (Estância Domain, foreland basin)", age_start=488, age_end=443,
         category="depositional", ref="Campos Neto et al., 2007; Costa et al., 2007a,b"),
    dict(label="Juá graben infill (orogenic collapse, Macururé Domain)", age_start=488, age_end=443,
         category="depositional", ref="Campos Neto et al., 2007; Costa et al., 2007a,b"),
    dict(label="Afro-Brazilian Depression", age_start=443, age_end=252,
         category="depositional", ref="Costa et al., 2007a,b; Silva et al., 2007"),
    dict(label="Pre-rift fluvio-eolian/lacustrine seq.", age_start=150, age_end=140,
         category="depositional", ref="Silva et al., 2007; Costa et al., 2007a,b; Campos Neto et al., 2007"),
    dict(label="Syn-rift lacustrine sequence", age_start=140, age_end=121,
         category="depositional", ref="Silva et al., 2007; Costa et al., 2007a,b; Campos Neto et al., 2007"),
    dict(label="Transitional evaporitic-carbonate seq.", age_start=121, age_end=100,
         category="depositional", ref="Silva et al., 2007; Costa et al., 2007a,b; Campos Neto et al., 2007"),
    dict(label="Drift onset: carbonate platform (Fm. Riachuelo)", age_start=100, age_end=94,
         category="depositional", ref="Campos Neto et al., 2007"),
    dict(label="Drift marine shale seq. (Fm. Cotinguiba)", age_start=94, age_end=66,
         category="depositional", ref="Campos Neto et al., 2007"),
    dict(label="Paleogene-Neogene marine/deltaic seq.", age_start=66, age_end=10,
         category="depositional", ref="Campos Neto et al., 2007; Silva et al., 2007"),
    dict(label="Barreiras Fm. (post-rift continental cover)", age_start=10, age_end=2,
         category="depositional", ref="Campos Neto et al., 2007; Silva et al., 2007; Costa et al., 2007a,b"),
    dict(label="Paraná-Etendeka LIP (Tristan da Cunha plume)", age_start=135, age_end=135,
         category="magmatic", ref="Zalán et al., 1991; Meisling et al., 2001; Jelinek, 2019; Frizon de Lamotte et al., 2015"),
    dict(label="RTJ-SEAL incipient rift (single branch, S→N propagation)", age_start=142, age_end=130,
         category="tectonic", ref="Costa et al., 2007a,b; Silva et al., 2007; Campos Neto et al., 2007; Tavares et al., 2026"),
    dict(label="Extension direction rotates E-W → NW-SE", age_start=140, age_end=130,
         category="tectonic", ref="Tavares et al., 2026"),
    dict(label="Rift propagation blocked at PeSZ", age_start=121, age_end=121,
         category="tectonic", ref="Tavares et al., 2026"),
    dict(label="RTJ rifting stalls, abandoned (failed rift)", age_start=121, age_end=113,
         category="tectonic", ref="Costa et al., 2007b"),
    dict(label="Sergipe sub-basin: amagmatic→magmatic transition, SDR volcanism", age_start=116, age_end=96,
         category="magmatic", ref="Silva et al., 2022"),
    dict(label="Stress field forced eastward; final SEAL-Gabon breakup", age_start=100, age_end=85,
         category="tectonic", ref="Tavares et al., 2026"),
    dict(label="Continental breakup / COB (Sergipe sub-basin, end-Albian)", age_start=100, age_end=100,
         category="tectonic", ref="Silva et al., 2022"),
    dict(label="Sergipe Microplate amalgamation (Arcoverde wedge resistance)", age_start=113, age_end=113,
         category="tectonic", ref="Szatmari and Milani, 1999"),
    dict(label="Cenomanian tectonic inversion (Mid-Atlantic Ridge push / Andean far-field stress)",
         age_start=100, age_end=94,
         category="tectonic", ref="Vasconcelos et al., 2019"),
    dict(label="Cratonic (W) cooling: end of Gondwanide orogeny", age_start=350, age_end=180,
         category="thermochron", ref="Jelinek et al., 2014, 2020"),
    dict(label="Southern VBF block: onset of cooling* (max. paleotemp., Cambrian-Ordovician)", age_start=541, age_end=444,
         category="thermochron", ref="Jelinek et al., 2020"),
    dict(label="Northern VBF block: onset of cooling* (max. paleotemp., Permian-Triassic)", age_start=299, age_end=201,
         category="thermochron", ref="Jelinek et al., 2020"),
    dict(label="Post-breakup denudation pulse (passive margin + RTJ shoulders)", age_start=140, age_end=120,
         category="thermochron", ref="Harman et al., 1998; Turner et al., 2008; Jelinek et al., 2014"),
    dict(label="AFT age clusters / exhumation (1–3 km)", age_start=100, age_end=90,
         category="thermochron", ref="Morais Neto et al., 2009; Japsen et al., 2012"),
    dict(label="Campanian regional cooling pulse", age_start=80, age_end=75,
         category="thermochron", ref="Jelinek et al., 2014"),
    dict(label="PeSZ: young AFT ages (localized denudation)", age_start=85, age_end=76,
         category="thermochron", ref="Harman et al., 1998"),
    dict(label="Neogene accelerated cooling (semi-arid transition)", age_start=20, age_end=0,
         category="thermochron", ref="Morais Neto et al., 2009; Japsen et al., 2012; Jelinek et al., 2014"),
]

CATEGORY_COLORS = {
    "tectonic": "#8e5a2f",
    "depositional": "#c9a13b",
    "magmatic": "#a6423a",
    "thermochron": "#3f6f8f",
}
CATEGORY_LABELS = {
    "tectonic": "Tectonic event",
    "depositional": "Depositional event",
    "magmatic": "Magmatic event",
    "thermochron": "Cooling episodes",
}

GEO_PERIODS = [
    ("Cm", 541, 485), ("O", 485, 444), ("S", 444, 419),
    ("D", 419, 359), ("C", 359, 299), ("P", 299, 252),
    ("T", 252, 201), ("J", 201, 145), ("K", 145, 66),
    ("Pg", 66, 23), ("N", 23, 2.6), ("Q", 2.6, 0),
]

ICS_PERIOD_COLORS = {
    "Cm": "#7FA056", "O": "#009270", "S": "#B3E1B6", "D": "#CB8C37",
    "C": "#67A599", "P": "#F04028", "T": "#812B92", "J": "#34B2C9",
    "K": "#7FC64E", "Pg": "#FD9A52", "N": "#FFE619", "Q": "#F9F97F",
}
PERIOD_BAND_ALPHA = 0.30
XMAX = 545


def split_outside_parens(s, sep=";"):
    parts, depth, current = [], 0, ""
    for ch in s:
        if ch == "(":
            depth += 1
            current += ch
        elif ch == ")":
            depth -= 1
            current += ch
        elif ch == sep and depth == 0:
            parts.append(current.strip())
            current = ""
        else:
            current += ch
    if current.strip():
        parts.append(current.strip())
    return parts


def extract_citation_keys(ref_string):
    keys = []
    for part in split_outside_parens(ref_string):
        m = re.match(r"^(.*?)\s*\([^()]*\)\s*$", part)
        main = m.group(1).strip() if m else part.strip()
        m2 = re.match(r"^(.*?),\s*((?:\d{4}(?:\s*,\s*\d{4})*))$", main)
        if m2 and "," in m2.group(2):
            author = m2.group(1).strip()
            years = [y.strip() for y in m2.group(2).split(",")]
            keys.extend(f"{author}, {y}" for y in years)
        else:
            keys.append(main)
    return keys


def build_reference_registry(events):
    registry = {}
    for ev in events:
        for key in extract_citation_keys(ev["ref"]):
            if key not in registry:
                registry[key] = len(registry) + 1
    return registry


def ref_numbers_str(ref_string, registry):
    nums = sorted(registry[k] for k in extract_citation_keys(ref_string))
    return ",".join(str(n) for n in nums)


def reference_list_text(registry):
    ordered = sorted(registry.items(), key=lambda kv: kv[1])
    return "  ".join(f"[{n}] {key}." for key, n in ordered)


def pack_bar_lanes(events, n_lanes, pad_ma=3.0):
    events_sorted = sorted(events, key=lambda e: -e["age_start"])
    occupied_min = [np.inf] * n_lanes
    lanes = {}
    for ev in events_sorted:
        ev_max = max(ev["age_start"], ev["age_end"]) + pad_ma
        ev_min = min(ev["age_start"], ev["age_end"]) - pad_ma
        sub = None
        for i in range(n_lanes):
            if ev_max <= occupied_min[i]:
                sub = i
                break
        if sub is None:
            sub = int(np.argmax(occupied_min))
        occupied_min[sub] = ev_min
        lanes[id(ev)] = sub
    return lanes


def wrap_event_text(ev, ref_registry, width=42):
    full = f"{ev['label']} [{ref_numbers_str(ev['ref'], ref_registry)}]"
    return textwrap.wrap(full, width=width, break_long_words=False) or [full]


def pack_label_tiers(events, ref_registry, char_width_ma=3.0, pad_ma=5.0, wrap_width=42):
    events_sorted = sorted(events, key=lambda e: -e["age_start"])
    tier_free_until = []
    tiers, lines_by_ev = {}, {}
    for ev in events_sorted:
        cx = (ev["age_start"] + ev["age_end"]) / 2
        lines = wrap_event_text(ev, ref_registry, width=wrap_width)
        lines_by_ev[id(ev)] = lines
        max_chars = max(len(ln) for ln in lines)
        half_w = max_chars * char_width_ma / 2 + pad_ma
        left_edge, right_edge = cx + half_w, cx - half_w
        placed = False
        for i, free_until in enumerate(tier_free_until):
            if left_edge <= free_until:
                tiers[id(ev)] = i
                tier_free_until[i] = right_edge
                placed = True
                break
        if not placed:
            tiers[id(ev)] = len(tier_free_until)
            tier_free_until.append(right_edge)
    return tiers, lines_by_ev, len(tier_free_until)


def layout_category(events, n_bar_lanes, ref_registry, bar_lane_h=0.6, tier_h=1.05, wrap_width=42):
    lanes = pack_bar_lanes(events, n_bar_lanes)
    tiers, lines_by_ev, n_tiers = pack_label_tiers(events, ref_registry, wrap_width=wrap_width)
    bar_zone_h = n_bar_lanes * bar_lane_h
    label_zone_h = n_tiers * tier_h
    return dict(lanes=lanes, tiers=tiers, lines_by_ev=lines_by_ev, n_tiers=n_tiers,
                bar_zone_h=bar_zone_h, label_zone_h=label_zone_h,
                bar_lane_h=bar_lane_h, tier_h=tier_h, n_bar_lanes=n_bar_lanes)


def draw_event_row(ax, events, category, y0, layout):
    color = CATEGORY_COLORS[category]
    bar_lane_h, tier_h = layout["bar_lane_h"], layout["tier_h"]
    y_label_zone_bottom = y0 + layout["bar_zone_h"]
    for ev in events:
        lane = layout["lanes"][id(ev)]
        bar_y = y0 + lane * bar_lane_h
        bar_cy = bar_y + bar_lane_h * 0.5
        cx = (ev["age_start"] + ev["age_end"]) / 2
        if ev["age_start"] == ev["age_end"]:
            ax.plot(ev["age_start"], bar_cy, marker="D", color=color, markersize=6,
                     zorder=4, markeredgecolor="white", markeredgewidth=0.6)
        else:
            ax.barh(bar_y + bar_lane_h * 0.12, width=ev["age_end"] - ev["age_start"],
                     left=ev["age_start"], height=bar_lane_h * 0.76, color=color,
                     edgecolor="white", linewidth=0.5, zorder=3, align="edge")
        tier = layout["tiers"][id(ev)]
        lines = layout["lines_by_ev"][id(ev)]
        label_y = y_label_zone_bottom + tier * tier_h + 0.12
        ax.plot([cx, cx], [bar_cy, label_y], color=color, lw=0.7, alpha=0.75, zorder=2)
        ax.plot(cx, bar_cy, marker="o", color=color, markersize=2.6, zorder=4, markeredgecolor="none")
        ax.text(cx, label_y, "\n".join(lines), fontsize=7.3, ha="center", va="bottom",
                 color="#222222", zorder=5, linespacing=1.2)
    y_top = y_label_zone_bottom + layout["n_tiers"] * tier_h + 0.15
    return y0, y_top


def numeric(series):
    s = series.astype(str).str.strip()
    s = s.replace({"-": None, "--": None, "": None, "nan": None, "NaN": None})
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def load_literature_ages(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xlsm", ".xls"):
        df = pd.read_excel(path, sheet_name=SHEET_NAME)
    elif ext == ".csv":
        df = pd.read_csv(path)
    elif ext in (".tsv", ".txt"):
        df = pd.read_csv(path, sep="\t")
    else:
        raise ValueError(f"Unsupported extension: {ext} (use .xlsx, .csv or .tsv)")

    df.columns = [str(c).strip().lower() for c in df.columns]
    alias_lower = {k.strip().lower(): v for k, v in COLUMN_ALIASES.items()}
    aliased_targets = set(alias_lower.values())
    for col in list(df.columns):
        if col in aliased_targets and col not in alias_lower:
            df = df.drop(columns=[col])
    df = df.rename(columns=alias_lower)

    required = ["age_ma", "domain"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}. Found: {list(df.columns)}")

    df["age_ma"] = numeric(df["age_ma"])
    dom_key = df["domain"].astype(str).str.strip().str.lower()
    df["domain_group"] = dom_key.map(DOMAIN_ALIASES)
    df = df.dropna(subset=["age_ma", "domain_group"]).reset_index(drop=True)
    keep_cols = ["age_ma", "domain_group"] + [c for c in ["source", "sample_id"] if c in df.columns]
    out = df[keep_cols].rename(columns={"domain_group": "domain"})
    return out


def main():
    categories = ["depositional", "magmatic", "tectonic", "thermochron"]
    group_of_cat = {
        "thermochron": "Regional cooling episodes (AFT, literature)",
        "tectonic": "Geodynamic events",
        "magmatic": "Geodynamic events",
        "depositional": "Sedimentation events",
    }
    n_bar_lanes = {"depositional": 2, "magmatic": 1, "tectonic": 2, "thermochron": 2}

    ref_registry = build_reference_registry(PHANEROZOIC_EVENTS)
    events_by_cat = {cat: [ev for ev in PHANEROZOIC_EVENTS if ev["category"] == cat] for cat in categories}

    bar_lane_h = 0.6
    tier_h = 1.05
    layouts = {cat: layout_category(events_by_cat[cat], n_bar_lanes[cat], ref_registry,
                                     bar_lane_h=bar_lane_h, tier_h=tier_h)
               for cat in categories}

    cat_gap = 1.35
    y_base = {}
    cursor = 0.0
    for cat in categories:
        y_base[cat] = cursor
        cursor += layouts[cat]["bar_zone_h"] + layouts[cat]["label_zone_h"] + cat_gap
    top_data = cursor - cat_gap + 0.3

    note1 = (
        "* Onset-of-cooling age = modelled maximum-paleotemperature timing (t-T thermal history modelling), "
        "with an open (undetermined) lower age bound in the source [see reference list] - not a central AFT age."
    )
    note2 = (
        "VBF = Vaza-Barris (River) Fault system; PeSZ = Pernambuco Shear Zone; RTJ = Reconcavo-Tucano-Jatoba rift system; "
        "SEAL = Sergipe-Alagoas Basin."
    )
    wrap_width = 200
    note1_wrapped = "\n".join(textwrap.wrap(note1, width=wrap_width))
    note2_wrapped = "\n".join(textwrap.wrap(note2, width=wrap_width))
    ref_list_wrapped = "\n".join(textwrap.wrap(reference_list_text(ref_registry), width=wrap_width))

    line_in = 0.135
    block_gap_in = 0.10
    title_line_in = 0.145

    footer_in = 0.12 + (note1_wrapped.count("\n") + 1) * line_in + block_gap_in
    footer_in += (note2_wrapped.count("\n") + 1) * line_in + block_gap_in
    footer_in += title_line_in + 0.03
    footer_in += (ref_list_wrapped.count("\n") + 1) * line_in
    footer_in += 0.15

    category_legend_in = 0.85
    in_per_data_unit = 0.50
    top_content_in = top_data * in_per_data_unit
    title_in = 1.15

    fig_h = title_in + top_content_in + category_legend_in + footer_in
    fig_w = 17.0

    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h))
    fig.patch.set_facecolor("white")

    for name, t0, t1 in GEO_PERIODS:
        ax.axvspan(t0, t1, color=ICS_PERIOD_COLORS[name], alpha=PERIOD_BAND_ALPHA, zorder=0, lw=0)

    for cat in categories:
        draw_event_row(ax, events_by_cat[cat], cat, y_base[cat], layouts[cat])

    ax.set_ylim(-0.4, top_data)
    yticks, yticklabels = [], []
    for cat in categories:
        yticks.append(y_base[cat] + layouts[cat]["bar_zone_h"] / 2)
        yticklabels.append(CATEGORY_LABELS[cat])
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels, fontsize=9.5, style="italic")
    ax.tick_params(axis="y", length=0)
    ax.spines[["top", "right", "left"]].set_visible(False)

    group_span = {}
    for cat in categories:
        g = group_of_cat[cat]
        y0 = y_base[cat] - cat_gap * 0.4
        y1 = y_base[cat] + layouts[cat]["bar_zone_h"] + layouts[cat]["label_zone_h"] + cat_gap * 0.4
        if g not in group_span:
            group_span[g] = [y0, y1]
        else:
            group_span[g][0] = min(group_span[g][0], y0)
            group_span[g][1] = max(group_span[g][1], y1)

    yaxis_blend = ax.get_yaxis_transform()
    for g, (y0, y1) in group_span.items():
        ax.text(-0.075, (y0 + y1) / 2, g, transform=yaxis_blend,
                 fontsize=9.5, fontweight="bold", color="#333333",
                 ha="center", va="center", rotation=90, clip_on=False)
    for i in range(len(categories) - 1):
        c_below, c_above = categories[i], categories[i + 1]
        if group_of_cat[c_below] != group_of_cat[c_above]:
            y_div = y_base[c_above] - cat_gap * 0.5
            ax.axhline(y_div, color="#999999", lw=0.8, ls=(0, (4, 3)), zorder=1)

    ax.set_xlim(XMAX, -35)
    ax.set_xlabel("Age (Ma)", fontsize=11)

    for name, t0, t1 in GEO_PERIODS:
        if name == "Cm":
            ax.text((t0 + t1) / 2, top_data * 0.99, "Ꞓ", fontsize=9, ha="center", va="top",
                     color="#555555", fontfamily="FreeSerif")
        else:
            ax.text((t0 + t1) / 2, top_data * 0.99, name, fontsize=7.5, ha="center", va="top",
                     color="#555555", style="italic")

    fig.suptitle("Phanerozoic regional geology context - São Francisco Craton / Sergipano Belt boundary\n"
                 "Cooling episodes, geodynamic events and sedimentation (compiled from regional literature)",
                 fontsize=13, fontweight="bold", y=1.0 - (0.10 / fig_h))

    bottom_frac = footer_in / fig_h
    handles = [mpatches.Patch(color=CATEGORY_COLORS[c], label=CATEGORY_LABELS[c]) for c in categories]
    fig.legend(handles=handles, loc="lower center", fontsize=9, frameon=False, ncol=4,
               bbox_to_anchor=(0.5, bottom_frac + 0.18 / fig_h))

    y_in = footer_in - 0.12

    def place(text, fontsize, weight="normal", color="#444444", style="normal", gap_after_in=0.0):
        nonlocal y_in
        fig.text(0.5, y_in / fig_h, text, ha="center", va="top", fontsize=fontsize,
                  fontweight=weight, color=color, style=style)
        n_lines = text.count("\n") + 1
        y_in -= n_lines * line_in + gap_after_in

    place(note1_wrapped, 7.3, color="#444444", style="italic", gap_after_in=block_gap_in)
    place(note2_wrapped, 6.2, color="#666666", style="italic", gap_after_in=block_gap_in)
    place("References:", 6.5, weight="bold", color="#444444", gap_after_in=0.03)
    place(ref_list_wrapped, 5.5, color="#555555")

    fig.subplots_adjust(left=0.075, right=0.985, top=1.0 - title_in / fig_h,
                         bottom=bottom_frac + category_legend_in / fig_h)

    fig.savefig("phanerozoic_summary.png", dpi=220)
    fig.savefig("phanerozoic_summary.pdf")
    plt.show()


if __name__ == "__main__":
    main()


## 3. Mean track length (MTL) histograms per sample — Supplementary Fig. S2

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import gaussian_kde

fm.findSystemFonts(fontpaths=None, fontext='ttf')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Montserrat', 'DejaVu Sans', 'Arial']
plt.rcParams['pdf.fonttype'] = 42

INPUT_FILES = [
    "TFS-001.xlsx", "TFS-005.xlsx", "TFS-009.xlsx", "TFS-010.xlsx",
    "TFS-016.xlsx", "TFS-018.xlsx", "TFS-021.xlsx", "TFS-029.xlsx",
    "TFS-035.xlsx", "TFS-039.xlsx", "TFS-046.xlsx", "TFS-049.xlsx",
    "TFS-051.xlsx", "TFS-053.xlsx", "TFS-055.xlsx", "TFS-061.xlsx",
    "TFS-068.xlsx", "TFS-074.xlsx", "TFS-082.xlsx", "TFS-101.xlsx",
    "TFS-104.xlsx", "TFS-106.xlsx", "TFS-108.xlsx", "TFS-112.xlsx",
    "TFS-114.xlsx", "TFS-116.xlsx", "TFS-120.xlsx",
]
SHEET_NAME = "Length and angle"
DATA_COLUMN = "Length"

for file_path in INPUT_FILES:
    sample_name = os.path.splitext(os.path.basename(file_path))[0]
    print(f"Processing sample: {sample_name}")

    df = pd.read_excel(file_path, sheet_name=SHEET_NAME, header=1)
    data = df[DATA_COLUMN]

    n = len(data)
    mean_val = np.mean(data)
    std_val = np.std(data)
    mean_rounded = round(mean_val, 1)
    std_rounded = round(std_val, 1)

    fig = plt.figure(figsize=(8, 6))
    bin_edges = list(range(0, 19, 1))
    plt.hist(data, bins=bin_edges, color='#F5F5DC', alpha=1.0, edgecolor='black')

    kde = gaussian_kde(data)
    x_grid = np.linspace(data.min(), data.max(), 1000)
    density = kde(x_grid)
    bin_width = bin_edges[1] - bin_edges[0]
    scale = n * bin_width
    plt.plot(x_grid, density * scale, color='#8B0000', linewidth=2.5)

    plt.xlabel('Confined track length (μm)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(sample_name, fontsize=20, loc='left', fontweight='bold')

    plt.xlim(0, 18)
    even_ticks = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
    odd_ticks = [1, 3, 5, 7, 9, 11, 13, 15, 17]
    plt.gca().set_xticks(even_ticks)
    plt.gca().set_xticklabels(even_ticks, fontsize=11)
    plt.gca().set_xticks(odd_ticks, minor=True)
    plt.gca().tick_params(axis='x', which='minor', length=4, color='black')
    plt.gca().tick_params(axis='x', which='major', length=6, color='black')

    stats_text = f"{sample_name}\nn = {n}\nMTL = {mean_rounded}\n$\\sigma$ = {std_rounded}"
    plt.text(x=0.02, y=0.95, s=stats_text, transform=plt.gca().transAxes,
              fontsize=12, fontweight='bold', va='top', ha='left',
              bbox=dict(facecolor='white', edgecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(f'{sample_name}_histogram.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{sample_name}_histogram.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 4. AFT age vs. elevation — Supplementary Fig. S3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

INPUT_FILE = "samples_data.xlsx"

fm.findSystemFonts(fontpaths=None, fontext='ttf')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Montserrat', 'DejaVu Sans', 'Arial']

df = pd.read_excel(INPUT_FILE)
df.columns = df.columns.str.strip()

COL_SAMPLE = 'Sample'
COL_AGE = 'Central age (Ma)'
COL_ERR = 'Error (Ma)'
COL_ELEV = 'Elevation (m)'
COL_REGION = 'East/West Tucano Basin'
COL_UNIT = 'Unit'
COL_ROBUSTNESS = 'Robustness'

for col in [COL_SAMPLE, COL_REGION, COL_UNIT, COL_ROBUSTNESS]:
    df[col] = df[col].astype(str).str.strip()
for col in [COL_AGE, COL_ERR, COL_ELEV]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df[COL_AGE].notna() & (df[COL_AGE] > 0) & (df[COL_AGE] < 1000)]
df = df[df[COL_ELEV].notna() & (df[COL_ELEV] > -5000) & (df[COL_ELEV] < 5000)]
df = df.replace([np.inf, -np.inf], np.nan)

UNIT_COLORS = {
    'Itabaiana Dome': '#6959CD',
    'Macururé Domain': '#FF1493',
    'Poço Redondo Domain': '#F08080',
    'Arapiraca Complex': '#D8BFD8',
    'Canindé Domain': '#DC143C',
    'São Francisco Craton': '#BA55D3',
    'Pernambuco-Alagoas Terrane': '#A52A2A',
    'Jirau do Ponciano Dome': '#836FFF',
}

fig, ax = plt.subplots(figsize=(12, 8))

for _, row in df.iterrows():
    x = row[COL_AGE]
    x_err = row[COL_ERR]
    y = row[COL_ELEV]
    label = row[COL_SAMPLE]
    region = row[COL_REGION]
    unit = row[COL_UNIT]
    robust = row[COL_ROBUSTNESS]

    color = UNIT_COLORS.get(unit, 'gray')
    marker = 's' if region == 'East' else 'o'
    size = 8

    if robust.lower() == 'no':
        ax.scatter(x, y, marker=marker, s=size**2, facecolor='none', edgecolor='black',
                   linewidth=1.0, zorder=3, linestyle='--')

    ax.errorbar(x, y, xerr=None if pd.isna(x_err) else x_err, yerr=None,
                fmt=marker, color=color, ecolor='#A0A0A0',
                capsize=3, elinewidth=1.0, markeredgewidth=1.0, markersize=size, zorder=2)

    ax.text(x + 4, y + 0.02, label, fontsize=10, fontweight='normal', va='bottom', ha='left')

ax.set_xlabel('AFT Central Age (Ma)', fontsize=14, fontweight='bold')
ax.set_ylabel('Elevation (m)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 450)
ax.set_ylim(-100, 1000)
ax.set_xticks(np.arange(0, 451, 50))

east_patch = mlines.Line2D([], [], color='white', marker='s', linestyle='None',
                            markersize=10, markeredgecolor='black', markeredgewidth=1.5, label='East RTJ rift')
west_patch = mlines.Line2D([], [], color='white', marker='o', linestyle='None',
                            markersize=10, markeredgecolor='black', markeredgewidth=1.5, label='West RTJ rift')

geo_units = ['São Francisco Craton', 'Itabaiana Dome', 'Jirau do Ponciano Dome',
             'Arapiraca Complex', 'Pernambuco-Alagoas Terrane', 'Poço Redondo Domain']
geo_patches = [
    mlines.Line2D([], [], color=UNIT_COLORS[u], marker='s', linestyle='None',
                  markersize=10, markeredgecolor='black', label=u)
    for u in geo_units
]
brasiliano_patches = [
    mlines.Line2D([], [], color=UNIT_COLORS['Macururé Domain'], marker='s', linestyle='None',
                  markersize=10, markeredgecolor='black', label='Macururé Domain'),
    mlines.Line2D([], [], color=UNIT_COLORS['Canindé Domain'], marker='s', linestyle='None',
                  markersize=10, markeredgecolor='black', label='Canindé Domain'),
]
insufficient_patch = mpatches.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='black',
                                         linewidth=1.0, linestyle='--',
                                         label='insufficient track lengths (n<50)')

handles = [east_patch, west_patch] + geo_patches + brasiliano_patches + [insufficient_patch]
ax.legend(handles=handles, loc='upper right', fontsize=10, framealpha=0.9)

plt.rcParams['pdf.fonttype'] = 42
fig.savefig('elevation_age_plot.pdf', bbox_inches='tight')
fig.tight_layout()
plt.show()


## 5. Boomerang plot (MTL vs. central age), measured and projected — Supplementary Fig. S4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

INPUT_FILE = "samples_data.xlsx"

fm.findSystemFonts(fontpaths=None, fontext='ttf')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Montserrat', 'DejaVu Sans', 'Arial']
plt.rcParams['pdf.fonttype'] = 42

UNIT_COLORS = {
    'Itabaiana Dome': '#6959CD',
    'Macururé Domain': '#FF1493',
    'Poço Redondo Domain': '#F08080',
    'Arapiraca Complex': '#D8BFD8',
    'Canindé Domain': '#DC143C',
    'São Francisco Craton': '#BA55D3',
    'Pernambuco-Alagoas Terrane': '#A52A2A',
    'Jirau do Ponciano Dome': '#836FFF',
}

COL_SAMPLE = 'Sample'
COL_AGE = 'Central age (Ma)'
COL_ERR = 'Error (Ma)'
COL_REGION = 'East/West Tucano Basin'
COL_UNIT = 'Unit'
COL_ROBUSTNESS = 'Robustness'


def make_boomerang_plot(mtl_column, ylabel, ylim, out_basename):
    df = pd.read_excel(INPUT_FILE)
    df.columns = df.columns.str.strip()

    for col in [COL_SAMPLE, COL_REGION, COL_UNIT, COL_ROBUSTNESS]:
        df[col] = df[col].astype(str).str.strip()
    for col in [COL_AGE, COL_ERR, mtl_column]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df[df[COL_AGE].notna() & (df[COL_AGE] > 0) & (df[COL_AGE] < 1000)]
    df = df[df[mtl_column].notna() & (df[mtl_column] > 0) & (df[mtl_column] < 20)]
    df = df.replace([np.inf, -np.inf], np.nan)

    fig, ax = plt.subplots(figsize=(12, 8))

    for _, row in df.iterrows():
        x = row[COL_AGE]
        x_err = row[COL_ERR]
        y = row[mtl_column]
        label = row[COL_SAMPLE]
        region = row[COL_REGION]
        unit = row[COL_UNIT]
        robust = row[COL_ROBUSTNESS]

        color = UNIT_COLORS.get(unit, 'gray')
        marker = 's' if region == 'East' else 'o'
        size = 8

        if robust.lower() == 'no':
            ax.scatter(x, y, marker=marker, s=size**2, facecolor='none', edgecolor='black',
                       linewidth=1.0, zorder=3, linestyle='--')

        ax.errorbar(x, y, xerr=None if pd.isna(x_err) else x_err, yerr=None,
                    fmt=marker, color=color, ecolor='#A0A0A0',
                    capsize=3, elinewidth=1.0, markeredgewidth=1.0, markersize=size, zorder=2)

        ax.text(x + 4, y + 0.02, label, fontsize=10, fontweight='normal', va='bottom', ha='left')

    ax.set_xlabel('AFT Central Age (Ma)', fontsize=14, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
    ax.set_xlim(0, 450)
    ax.set_ylim(*ylim)
    ax.set_xticks(np.arange(0, 451, 50))

    east_patch = mlines.Line2D([], [], color='white', marker='s', linestyle='None',
                                markersize=10, markeredgecolor='black', markeredgewidth=1.5, label='East RTJ rift')
    west_patch = mlines.Line2D([], [], color='white', marker='o', linestyle='None',
                                markersize=10, markeredgecolor='black', markeredgewidth=1.5, label='West RTJ rift')
    geo_units = ['São Francisco Craton', 'Itabaiana Dome', 'Jirau do Ponciano Dome',
                 'Arapiraca Complex', 'Pernambuco-Alagoas Terrane', 'Poço Redondo Domain']
    geo_patches = [
        mlines.Line2D([], [], color=UNIT_COLORS[u], marker='s', linestyle='None',
                      markersize=10, markeredgecolor='black', label=u)
        for u in geo_units
    ]
    brasiliano_patches = [
        mlines.Line2D([], [], color=UNIT_COLORS['Macururé Domain'], marker='s', linestyle='None',
                      markersize=10, markeredgecolor='black', label='Macururé Domain'),
        mlines.Line2D([], [], color=UNIT_COLORS['Canindé Domain'], marker='s', linestyle='None',
                      markersize=10, markeredgecolor='black', label='Canindé Domain'),
    ]
    insufficient_patch = mpatches.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='black',
                                             linewidth=1.0, linestyle='--',
                                             label='insufficient track lengths (n<50)')
    handles = [east_patch, west_patch] + geo_patches + brasiliano_patches + [insufficient_patch]
    ax.legend(handles=handles, loc='upper right', fontsize=10, framealpha=0.9)

    fig.savefig(f'{out_basename}.pdf', bbox_inches='tight')
    fig.tight_layout()
    plt.show()


make_boomerang_plot('MTL measured', 'MTL measured (μm)', (10.5, 14.2), 'boomerang_plot_measured')
make_boomerang_plot('MTL projected', 'MTL Projected (μm)', (13, 15), 'boomerang_plot_projected')


## 6. Expected t–T paths by crustal domain and MTL cluster — Figure 5 (main text)

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

INPUT_DIR = "tt_files"
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

fm.findSystemFonts(fontpaths=None, fontext="ttf")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Montserrat", "DejaVu Sans", "Arial"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

DOMAIN = {
    'TFS-101': 'cratonic', 'TFS-104': 'cratonic', 'TFS-106': 'cratonic', 'TFS-108': 'cratonic',
    'TFS-112': 'cratonic', 'TFS-114': 'cratonic', 'TFS-116': 'cratonic', 'TFS-120': 'cratonic',
    'TFS-082': 'transitional',
    'TFS-001': 'orogenic', 'TFS-005': 'orogenic', 'TFS-009': 'orogenic', 'TFS-010': 'orogenic',
    'TFS-016': 'orogenic', 'TFS-021': 'orogenic', 'TFS-035': 'orogenic', 'TFS-039': 'orogenic',
    'TFS-049': 'orogenic', 'TFS-051': 'orogenic', 'TFS-053': 'orogenic', 'TFS-055': 'orogenic',
    'TFS-061': 'orogenic', 'TFS-068': 'orogenic', 'TFS-074': 'orogenic',
}
MTL_CLUSTER = {
    'TFS-106': 'low', 'TFS-112': 'low', 'TFS-114': 'low', 'TFS-108': 'low',
    'TFS-061': 'high', 'TFS-055': 'high', 'TFS-053': 'high', 'TFS-016': 'high',
    'TFS-021': 'high', 'TFS-074': 'high',
}
HIGHLIGHT = {'TFS-104', 'TFS-108', 'TFS-082'}
AGE_OUTLIER = {'TFS-039'}

COLORS = {'low': '#4C78A8', 'high': '#E4572E', 'moderate': '#9CA3AF'}
DOMAIN_ORDER = ['cratonic', 'transitional', 'orogenic']
DOMAIN_TITLES = {'cratonic': 'Cratonic', 'transitional': 'Transitional', 'orogenic': 'Orogenic'}


def load_tt_path(path):
    rows = []
    with open(path, "r") as f:
        lines = f.readlines()
    for ln in lines[1:]:
        parts = ln.split()
        if len(parts) != 4:
            continue
        try:
            t, temp, lo, hi = (float(x) for x in parts)
        except ValueError:
            continue
        rows.append((t, temp, lo, hi))
    rows.sort(key=lambda r: -r[0])
    t = np.array([r[0] for r in rows])
    T = np.array([r[1] for r in rows])
    lo = np.array([r[2] for r in rows])
    hi = np.array([r[3] for r in rows])
    anchor = (t == t.max()) & (T == 0) & (lo == 0) & (hi == 0)
    return t[~anchor], T[~anchor]


files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.txt")))
data = {}
for fp in files:
    base = os.path.basename(fp)
    sample = base.split(" expected")[0]
    t, T = load_tt_path(fp)
    data[sample] = (t, T)

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True, sharex=True)
t_max_global = max(t.max() for t, T in data.values())

for ax, dom in zip(axes, DOMAIN_ORDER):
    samples = [s for s in DOMAIN if DOMAIN[s] == dom]
    ax.axvspan(150, 100, color="#DDDDDD", alpha=0.6, zorder=0)
    for s in samples:
        t, T = data[s]
        cluster = MTL_CLUSTER.get(s, "moderate")
        color = COLORS[cluster]
        is_highlighted = s in HIGHLIGHT
        lw = 2.6 if is_highlighted else 1.1
        alpha = 1.0 if is_highlighted else 0.75
        ls = "--" if s in AGE_OUTLIER else "-"
        zorder = 5 if is_highlighted else 2
        ax.plot(t, T, color=color, lw=lw, alpha=alpha, linestyle=ls, zorder=zorder)
        if is_highlighted:
            ax.annotate(s, (t[0], T[0]), fontsize=7.5, fontweight="bold",
                        color=color, xytext=(4, 2), textcoords="offset points")
        if s in AGE_OUTLIER:
            ax.annotate(s + " (age outlier)", (t[0], T[0]), fontsize=7.5, fontweight="bold",
                        color="#333333", xytext=(4, -8), textcoords="offset points")

    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.set_xlim(t_max_global * 1.03, -5)
    ax.set_title(f"{DOMAIN_TITLES[dom]}\n(n={len(samples)})", fontsize=11.5, fontweight="bold", loc="center")
    ax.set_xlabel("Time (Ma)", fontsize=11)
    ax.grid(alpha=0.15)

axes[0].set_ylabel("Temperature (°C)", fontsize=11)
axes[1].annotate("150-100 Ma\n(syn-rift)", xy=(125, 15), ha="center", fontsize=8, color="#666666")

legend_handles = [
    plt.Line2D([0], [0], color=COLORS['low'], lw=2, label='Low MTL cluster'),
    plt.Line2D([0], [0], color=COLORS['high'], lw=2, label='High MTL cluster'),
    plt.Line2D([0], [0], color=COLORS['moderate'], lw=2, label='Moderate / intermediate MTL'),
    plt.Line2D([0], [0], color="#333333", lw=2.6, label='Neogene-anomalous / highlighted sample'),
    plt.Line2D([0], [0], color="#333333", lw=1.5, linestyle="--", label='Age outlier (TFS-039)'),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=5, fontsize=8.5,
           frameon=False, bbox_to_anchor=(0.5, -0.06))

fig.suptitle("Expected Model t-T paths by domain and MTL cluster", fontsize=15, fontweight="bold", y=1.03, x=0.02, ha="left")

plt.tight_layout()
fig.subplots_adjust(wspace=0.08)

fig.savefig(os.path.join(OUTPUT_DIR, "domain_tT_panels.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(OUTPUT_DIR, "domain_tT_panels.pdf"), bbox_inches="tight")
plt.show()


## 7. QTQt cooling-rate and exhumation-rate calculation, per sample and combined — Supplementary Table S3 / Fig. S6

In [ ]:
import os
import re
import glob
import csv
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

INPUT_DIR = "tt_files"
FILE_PATTERN = "*expected Tt file.txt"
OUTPUT_DIR = os.path.join(INPUT_DIR, "cooling_exhumation_output")

GRADIENT_LOW_C_KM = 18.0
GRADIENT_MID_C_KM = 25.0
GRADIENT_HIGH_C_KM = 30.0

BREAKPOINTS_MA = [1000, 150, 100, 50, 20, 0]

fm.findSystemFonts(fontpaths=None, fontext="ttf")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Montserrat", "DejaVu Sans", "Arial"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_tt_path(path):
    rows = []
    with open(path, "r") as f:
        lines = f.readlines()
    for ln in lines[1:]:
        parts = ln.split()
        if len(parts) != 4:
            continue
        try:
            t, temp, lo, hi = (float(x) for x in parts)
        except ValueError:
            continue
        rows.append((t, temp, lo, hi))

    rows.sort(key=lambda r: -r[0])
    t = np.array([r[0] for r in rows])
    T = np.array([r[1] for r in rows])
    lo = np.array([r[2] for r in rows])
    hi = np.array([r[3] for r in rows])

    anchor = (t == t.max()) & (T == 0) & (lo == 0) & (hi == 0)
    t, T, lo, hi = t[~anchor], T[~anchor], lo[~anchor], hi[~anchor]
    return t, T, lo, hi


def segment_path(t, T, lo, hi, breakpoints_ma):
    bps = sorted(set(b for b in breakpoints_ma if t.min() <= b <= t.max()), reverse=True)
    if not bps or bps[0] != round(float(t.max())):
        bps = [t.max()] + [b for b in bps if b < t.max()]
    if bps[-1] != t.min():
        bps = bps + [t.min()] if bps[-1] > t.min() else bps

    segments = []
    for i in range(len(bps) - 1):
        t_hi, t_lo = bps[i], bps[i + 1]
        sel = (t <= t_hi) & (t >= t_lo)
        if sel.sum() < 2:
            continue
        t_seg, T_seg, lo_seg, hi_seg = t[sel], T[sel], lo[sel], hi[sel]

        a, b = np.polyfit(t_seg, T_seg, 1)
        cooling_rate = a
        T_pred = a * t_seg + b
        ss_res = np.sum((T_seg - T_pred) ** 2)
        ss_tot = np.sum((T_seg - T_seg.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        dt_seg = t_seg[0] - t_seg[-1]
        err_start = (hi_seg[0] - lo_seg[0]) / 2.0
        err_end = (hi_seg[-1] - lo_seg[-1]) / 2.0
        rate_err = np.sqrt(err_start**2 + err_end**2) / dt_seg if dt_seg > 0 else np.nan

        segments.append(dict(
            t_start=t_seg[0], t_end=t_seg[-1],
            T_start=T_seg[0], T_end=T_seg[-1],
            cooling_rate_c_myr=cooling_rate, cooling_rate_err_c_myr=rate_err, r2=r2,
        ))
    return segments


def add_exhumation_rates(segments):
    for s in segments:
        rate = s["cooling_rate_c_myr"]
        for label, grad in (("low", GRADIENT_HIGH_C_KM),
                             ("mid", GRADIENT_MID_C_KM),
                             ("high", GRADIENT_LOW_C_KM)):
            exh_km_myr = rate / grad if grad else np.nan
            s[f"exhumation_{label}_km_myr"] = exh_km_myr
            s[f"exhumation_{label}_m_myr"] = exh_km_myr * 1000.0
    return segments


def plot_sample(sample_name, t, T, lo, hi, segments, out_dir):
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(9, 9.5), sharex=True, gridspec_kw={"height_ratios": [2.3, 1]}
    )

    ax1.fill_between(t, lo, hi, color="#4C78A8", alpha=0.18, label="95% credible interval")
    ax1.plot(t, T, color="#1b1b1b", lw=2, label="Expected Model")

    colors = plt.cm.viridis(np.linspace(0.15, 0.9, max(len(segments), 1)))
    for s, c in zip(segments, colors):
        ax1.plot([s["t_start"], s["t_end"]], [s["T_start"], s["T_end"]],
                  color=c, lw=0, marker="o", ms=6, zorder=5)
        xm = (s["t_start"] + s["t_end"]) / 2
        ym = max(s["T_start"], s["T_end"]) + 4
        label = (f"{s['cooling_rate_c_myr']:.2f} °C/Myr\n"
                 f"{s['exhumation_mid_m_myr']:.1f} m/Myr")
        ax1.annotate(label, (xm, ym), ha="center", fontsize=8, color="#333333")

    ax1.invert_xaxis()
    ax1.invert_yaxis()
    ax1.set_ylabel("Temperature (°C)", fontsize=12)
    ax1.legend(loc="upper right", fontsize=8, frameon=True)
    ax1.grid(alpha=0.2)

    stats_text = (f"{sample_name}\nn segments = {len(segments)}\n"
                  f"oldest = {t.max():.0f} Ma\nyoungest = {t.min():.0f} Ma")
    ax1.text(x=0.98, y=0.02, s=stats_text, transform=ax1.transAxes,
              fontsize=10, fontweight="bold", va="bottom", ha="right",
              bbox=dict(facecolor="white", edgecolor="black", alpha=0.85, linewidth=0.5))

    ax1.set_title(sample_name, fontsize=20, loc="left", fontweight="bold")

    dT = -np.diff(T)
    dt = -np.diff(t)
    rate_point = dT / dt
    half_width = (hi - lo) / 2.0
    err_prop = np.sqrt(half_width[:-1] ** 2 + half_width[1:] ** 2) / dt

    ax2.axhline(0, color="grey", lw=0.7)
    ax2.plot(t[1:], rate_point, color="#B33951", lw=1)
    ax2.fill_between(t[1:], rate_point - err_prop, rate_point + err_prop, color="#B33951", alpha=0.15)
    ax2.set_xlabel("Time (Ma)", fontsize=12)
    ax2.set_ylabel("Cooling rate\n(°C/Myr)", fontsize=12)
    ax2.grid(alpha=0.2)

    plt.tight_layout()

    png_path = os.path.join(out_dir, f"{sample_name}_cooling_plot.png")
    pdf_path = os.path.join(out_dir, f"{sample_name}_cooling_plot.pdf")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.show()
    plt.close(fig)
    return png_path, pdf_path


def save_sample_csv(sample_name, segments, out_dir):
    path = os.path.join(out_dir, f"{sample_name}_cooling_exhumation.csv")
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "sample", "t_start_Ma", "t_end_Ma", "T_start_C", "T_end_C",
            "cooling_rate_C_per_Myr", "cooling_rate_err_C_per_Myr", "R2",
            f"exhumation_low_m_per_Myr(grad={GRADIENT_HIGH_C_KM}C/km)",
            f"exhumation_mid_m_per_Myr(grad={GRADIENT_MID_C_KM}C/km)",
            f"exhumation_high_m_per_Myr(grad={GRADIENT_LOW_C_KM}C/km)",
        ])
        for s in segments:
            w.writerow([
                sample_name, f"{s['t_start']:.1f}", f"{s['t_end']:.1f}",
                f"{s['T_start']:.2f}", f"{s['T_end']:.2f}",
                f"{s['cooling_rate_c_myr']:.4f}", f"{s['cooling_rate_err_c_myr']:.4f}", f"{s['r2']:.3f}",
                f"{s['exhumation_low_m_myr']:.2f}", f"{s['exhumation_mid_m_myr']:.2f}",
                f"{s['exhumation_high_m_myr']:.2f}",
            ])
    return path


def sample_name_from_path(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r"(.+?)[ _]?Tt[ _]?file$", stem, flags=re.IGNORECASE)
    return m.group(1).rstrip("_ ") if m else stem


def main():
    files = sorted(glob.glob(os.path.join(INPUT_DIR, FILE_PATTERN)))
    if not files:
        print(f"No files matching '{FILE_PATTERN}' found in: {INPUT_DIR}")
        return

    all_rows = []
    for path in files:
        sample_name = sample_name_from_path(path)
        print(f"Processing sample: {sample_name}")

        try:
            t, T, lo, hi = load_tt_path(path)
            if len(t) < 3:
                print(f"  skipped ({sample_name}): not enough data points")
                continue

            segments = segment_path(t, T, lo, hi, BREAKPOINTS_MA)
            segments = add_exhumation_rates(segments)

            save_sample_csv(sample_name, segments, OUTPUT_DIR)
            plot_sample(sample_name, t, T, lo, hi, segments, OUTPUT_DIR)

            for s in segments:
                row = {"sample": sample_name, **s}
                all_rows.append(row)

        except Exception as e:
            print(f"  ERROR processing {sample_name}: {e}")

    if all_rows:
        summary_path = os.path.join(OUTPUT_DIR, "ALL_SAMPLES_cooling_exhumation_summary.csv")
        keys = ["sample", "t_start", "t_end", "T_start", "T_end",
                "cooling_rate_c_myr", "cooling_rate_err_c_myr", "r2",
                "exhumation_low_m_myr", "exhumation_mid_m_myr", "exhumation_high_m_myr"]
        with open(summary_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            for row in all_rows:
                w.writerow({k: row.get(k) for k in keys})
        print(f"Combined summary saved: {summary_path}")

    print(f"Done. {len(files)} file(s) processed. Output folder: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()
